# Evaluate

Loads a saved checkpoint from `ModelRepository` and runs bootstrap evaluation
on the temporal test set and held-out symbols.

The split and raw data must be the same as when the model was trained so that
scalers are re-fitted identically.

Prerequisites:
- A checkpoint saved by `train.ipynb` (e.g. `data/models/universal_lstm_v1.pt`)
- `data/splits.yml` — the split used during training
- Same price / fundamental / sentiment data as training

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib.pyplot as plt

import src  # triggers load_dotenv()
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("evaluate")

## Load Checkpoint

In [ ]:
from src.repositories.models import ModelRepository
from src.model.lstm import SentimentLSTM
from src.model.transformer import SentimentTransformer

CHECKPOINT_NAME = "universal_lstm_v1"  # change to match saved name

model_repo = ModelRepository()
print(f"Available checkpoints: {model_repo.list()}")

ckpt = model_repo.load(CHECKPOINT_NAME)
cfg  = ckpt.config

if cfg["model_type"] == "lstm":
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_fundamentals=cfg["n_fundamentals"],
        n_sentiment_probs=cfg["n_sentiment_probs"],
    )
else:
    model = SentimentTransformer(
        n_factors=16,
        sentiment_dim=768,
        d_model=64,
        nhead=4,
        n_layers=6,
        dim_feedforward=128,
        dropout=0.2,
        n_fundamentals=cfg["n_fundamentals"],
        n_sentiment_probs=cfg["n_sentiment_probs"],
    )

model.load_state_dict(ckpt.state_dict)
model.eval()
print(f"Loaded {CHECKPOINT_NAME} — best_epoch={cfg.get('best_epoch')}, best_val_auc={cfg.get('best_val_auc', '?'):.4f}")

## Training History

In [ ]:
history = cfg.get("training_history")
if history:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, history["train_loss"], label="train")
    ax1.plot(epochs, history["val_loss"],   label="val")
    ax1.set(xlabel="epoch", ylabel="loss", title="Loss")
    ax1.legend()

    ax2.plot(epochs, history["val_auc"],      label="val AUC")
    ax2.plot(epochs, history["val_accuracy"], label="val accuracy")
    ax2.set(xlabel="epoch", ylabel="metric", title="Validation metrics")
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print("No training history in checkpoint.")

## Load Split + Data

Scalers are re-fitted from the same training data to guarantee they match the
checkpoint's stored scalers.

In [ ]:
from src.training import TrainingConfig, Split
from src.repositories.prices import PriceRepository
from src.repositories.fundamentals import FundamentalsRepository
from src.repositories.sentiment import SentimentRepository

PRICE_YEARS = [2018, 2019, 2020]  # must match training run

split = Split.load(Path("../data/splits.yml"))
print(f"Train: {len(split.train_symbols)}  Held-out: {len(split.held_out_symbols)}")

# --- prices ---
prices = PriceRepository()
price_dfs: dict = {}
for symbol in split.all_symbols:
    try:
        price_dfs[symbol] = prices.load_years(symbol, PRICE_YEARS)
    except FileNotFoundError:
        price_dfs[symbol] = None

# --- fundamentals ---
fund_repo = FundamentalsRepository()
fundamental_dfs = {s: fund_repo.load_history(s) for s in split.all_symbols}

# --- sentiment ---
sent_repo = SentimentRepository()
sentiment_dfs = {
    s: sent_repo.load(s) if sent_repo.exists(s) else None
    for s in split.all_symbols
}

print("Data loaded.")

## Build Datasets + DataLoaders

In [ ]:
from src.features.dataset import StockDataset, DataLoaderBuilder
from src.training import ComputeConfig, TrainingConfig

WINDOW = cfg["window"]

compute_config = ComputeConfig()  # cpu, no workers — inference only
config = TrainingConfig(window=WINDOW)

datasets: dict[str, StockDataset] = {}
for symbol in split.all_symbols:
    df = price_dfs.get(symbol)
    if df is None:
        continue
    try:
        datasets[symbol] = StockDataset(
            symbol=symbol,
            price_df=df,
            sentiment_df=sentiment_dfs.get(symbol),
            fundamental_df=fundamental_dfs.get(symbol),
            window=WINDOW,
            include_momentum_slope=True,
        )
    except RuntimeError as exc:
        logger.warning("Skipping %s: %s", symbol, exc)

builder = DataLoaderBuilder(datasets, split, config, compute_config)
_, _, test_loader = builder.build()  # fits scalers on training data
held_out_loader  = builder.build_held_out_loader()

print(f"Test batches    : {len(test_loader)}")
print(f"Held-out batches: {len(held_out_loader)}")

## Bootstrap Evaluation

In [ ]:
from src.model.trainer import Trainer

trainer = Trainer(model, config, compute_config)

r_test = trainer.bootstrap_evaluate(test_loader,     n_bootstrap=1000, seed=42)
r_ho   = trainer.bootstrap_evaluate(held_out_loader, n_bootstrap=1000, seed=42)


def _fmt(r, label: str) -> None:
    print(f"=== {label} ===")
    print(f"  AUC      : {r.auc_mean:.3f}  [{r.auc_ci_low:.3f}, {r.auc_ci_high:.3f}]")
    print(f"  Accuracy : {r.accuracy_mean:.3f}  [{r.accuracy_ci_low:.3f}, {r.accuracy_ci_high:.3f}]")
    print(f"  Precision: {r.precision_mean:.3f}  [{r.precision_ci_low:.3f}, {r.precision_ci_high:.3f}]")
    print(f"  Recall   : {r.recall_mean:.3f}  [{r.recall_ci_low:.3f}, {r.recall_ci_high:.3f}]")
    print(f"  N samples: {r.n_samples}")


_fmt(r_test, "Temporal test (training symbols, held-out time)")
print()
_fmt(r_ho,   "Held-out symbols (never seen during training)")

## Metric Summary Plot

In [ ]:
import numpy as np

metrics = ["AUC", "Accuracy", "Precision", "Recall"]
test_means  = [r_test.auc_mean, r_test.accuracy_mean, r_test.precision_mean, r_test.recall_mean]
test_lows   = [r_test.auc_ci_low, r_test.accuracy_ci_low, r_test.precision_ci_low, r_test.recall_ci_low]
test_highs  = [r_test.auc_ci_high, r_test.accuracy_ci_high, r_test.precision_ci_high, r_test.recall_ci_high]
ho_means    = [r_ho.auc_mean, r_ho.accuracy_mean, r_ho.precision_mean, r_ho.recall_mean]
ho_lows     = [r_ho.auc_ci_low, r_ho.accuracy_ci_low, r_ho.precision_ci_low, r_ho.recall_ci_low]
ho_highs    = [r_ho.auc_ci_high, r_ho.accuracy_ci_high, r_ho.precision_ci_high, r_ho.recall_ci_high]

x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

bars_t = ax.bar(x - w/2, test_means, w, label="Temporal test", color="steelblue")
bars_h = ax.bar(x + w/2, ho_means,   w, label="Held-out",      color="darkorange")

# 95 % CI error bars
ax.errorbar(
    x - w/2, test_means,
    yerr=[np.subtract(test_means, test_lows), np.subtract(test_highs, test_means)],
    fmt="none", color="black", capsize=4,
)
ax.errorbar(
    x + w/2, ho_means,
    yerr=[np.subtract(ho_means, ho_lows), np.subtract(ho_highs, ho_means)],
    fmt="none", color="black", capsize=4,
)

ax.set(xticks=x, xticklabels=metrics, ylim=(0.5, 1.0),
       ylabel="Score", title=f"Bootstrap evaluation — {CHECKPOINT_NAME}")
ax.legend()
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()